In [ ]:
"""Running a global fund

Demonstration of using LUSID to run funds fed from multiple source systems across multiple regions

Attributes
----------
instruments
transactions
cut labels
aggregation
recipes
quotes
cocoon
transaction configuration
"""

## 0) Import Libraries and Initialise LUSID Client

In [ ]:
# Import LUSID
import finbourne.sdk.services.lusid as lu
import finbourne.sdk.services.lusid.models as models
import lusid_sample_data as import_data
import globalfund as global_fund_tools
import finbourne_sdk_utils.cocoon.cocoon as cocoon_tools
from finbourne.sdk.extensions import SyncApiClientFactory, RefreshingToken

# Import Libraries
import pprint
from datetime import datetime, timedelta, time, date
import pytz
import uuid
import printer as prettyprint
from datetime import datetime
import pandas as pd
import numpy as np
import os
import json
globals = {}
                            
# Authenticate our user and create our API client
secrets_path = os.getenv("FBN_SECRETS_PATH")

api_factory = SyncApiClientFactory(
    access_token=RefreshingToken(),
    secrets_path=secrets_path,
    app_name="LusidJupyterNotebook")
    
print('LUSID Client Initialised')
print('LUSID version : ', api_factory.build(lu.ApplicationMetadataApi).get_lusid_versions().build_version)

## 1) Load Instrument Master across Regions

### Fetch the Instrument Master Data

#### US Instrument Master

In [ ]:
us_instrument_master = pd.read_csv("data/global-fund-US-instrument-master.csv")

us_instrument_master.head(n=20)

#### UK Instrument Master

In [ ]:
uk_instrument_master = pd.read_csv("data/global-fund-UK-instrument-master.csv")

uk_instrument_master.head(n=20)

#### Combined Instrument Master

In [ ]:
combined_instrument_master = pd.read_csv("data/global-fund-combined-instrument-master.csv")

combined_instrument_master.head(n=30)

### Load the Instruments into LUSID

In [ ]:
instrument_properties_scope = 'InstrumentProperties005'

instrument_identifier_mapping = {
    "Figi": "figi",
    "Isin": "isin",
    "ClientInternal": "client_internal"
}

instrument_mapping_required = {
    "name": "instrument_name"
}

instrument_mapping_optional = {}

responses = cocoon_tools.load_from_data_frame(
    api_factory=api_factory, 
    scope=instrument_properties_scope, 
    data_frame=combined_instrument_master, 
    mapping_required=instrument_mapping_required, 
    mapping_optional=instrument_mapping_optional, 
    file_type="instrument", 
    identifier_mapping=instrument_identifier_mapping, 
    property_columns=["s&p rating", "moodys_rating", "currency"])

prettyprint.instrument_response(responses["instruments"]["success"][0])

## 2) Load Transactions

### Create Transaction Portfolios

In [ ]:
scopes = {
    "UK": "UK_Thinkfolio",
    "US": "US_SimcorpDimension"
}

responses = global_fund_tools.create_portfolios(
    api_factory=api_factory, 
    scopes=list(scopes.values()), 
    code="GlobalCreditFund", 
    currency="EUR")

for response in responses:
    prettyprint.portfolio_response(response)

### Fetch the Transaction Data

#### US

In [ ]:
us_transactions = pd.read_csv("data/global-fund-us-transactions.csv")

us_transactions.head(n=20)

#### UK

In [ ]:
uk_transactions = pd.read_csv("data/global-fund-uk-transactions.csv")

uk_transactions.head(n=20)

#### Combined

In [ ]:
combined_transactions = pd.read_csv("data/global-fund-combined-transactions.csv")

for transaction_type in ["StartingBalance", "FundsIn"]:
    combined_transactions.loc[
        combined_transactions[
            "transaction_type"] == transaction_type, 'currency_transaction'] = combined_transactions["trade_currency"]
    
combined_transactions["exchange_rate"] = 1

combined_transactions.head(n=30)

#### Load the Transactions into LUSID

In [ ]:
transaction_field_mapping_required = {
    "code": "portfolio_code",
    "transaction_id": "id",
    "type": "transaction_type",
    "transaction_date": "transaction_date",
    "settlement_date": "settlement_date",
    "units": "units",
    "transaction_price.price": "transaction_price",
    "transaction_price.type": "$Price",
    "total_consideration.amount": "amount",
    "total_consideration.currency": "trade_currency",
    "transaction_currency": "trade_currency"
    }

transaction_field_mapping_optional = {
    "exchange_rate": "exchange_rate"
}

transaction_identifier_mapping = {
    "Figi": "figi",
    "Isin": "isin",
    "ClientInternal": "client_internal",
    "Currency": "currency_transaction"
}

for scope in list(scopes.values()):
    
    transaction_field_mapping_optional["source"] = f"${scope}"

    responses = cocoon_tools.load_from_data_frame(
        api_factory=api_factory, 
        scope=scope, 
        data_frame=combined_transactions.loc[
            combined_transactions["source"] == scope],
        mapping_required=transaction_field_mapping_required,
        mapping_optional=transaction_field_mapping_optional,
        identifier_mapping=transaction_identifier_mapping,
        file_type='transaction',
        property_columns=[
            "instrument_name", 
            "accounting_method",
            "mtom",
            "broker_executor",
            "location_region",
            "exposure_counterparty",
            "val",
            "compls",
            ])

    for response in responses['transactions']['success']:
        prettyprint.transactions_response(response, scope, response.href.split('/')[7])

## 3) Configure Transaction Types

### Get Unique Types

In [ ]:
combined_transactions.groupby(["transaction_type", "source"]).agg({"units": "max"})

### Create Configuration for Each Type

In [ ]:
movements1=[
    models.TransactionTypeMovement(
        movement_types='StockMovement',
        side='Side1',
        direction=1,
        properties=None,
        mappings=None),
    models.TransactionTypeMovement(
        movement_types='CashCommitment',
        side='Side2',
        direction=-1,
        properties=None,
        mappings=None)
]

response = api_factory.build(lu.TransactionConfigurationApi).set_transaction_type(
    source="US_SimcorpDimension",
    type="BY",
    transaction_type_request=models.TransactionTypeRequest(
        aliases=[
            models.TransactionTypeAlias(
                type="BY",
                description="BY",
                transaction_class="BY",
                transaction_roles='None',
            )
        ],
        movements=movements1,
    )
)

prettyprint.print_transaction_type(response)

response = api_factory.build(lu.TransactionConfigurationApi).set_transaction_type(
    source="UK_Thinkfolio",
    type="Purchase",
    transaction_type_request=models.TransactionTypeRequest(
        aliases=[
            models.TransactionTypeAlias(
                type="Purchase",
                description="Purchase",
                transaction_class="Purchase",
                transaction_roles='None',
            )
        ],
        movements=movements1,
    )
)
prettyprint.print_transaction_type(response)

movements2=[
    models.TransactionTypeMovement(
        movement_types='CashCommitment',
        side='Side2',
        direction=1,
        properties=None,
        mappings=None)
]

response = api_factory.build(lu.TransactionConfigurationApi).set_transaction_type(
    source="US_SimcorpDimension",
    type="FundsIn",
    transaction_type_request=models.TransactionTypeRequest(
        aliases=[
            models.TransactionTypeAlias(
                type="FundsIn",
                description="FundsIn",
                transaction_class="FundsIn",
                transaction_roles='None',
            )
        ],
        movements=movements2,
    )
)

prettyprint.print_transaction_type(response)

response = api_factory.build(lu.TransactionConfigurationApi).set_transaction_type(
    source="UK_Thinkfolio",
    type="StartingBalance",
    transaction_type_request=models.TransactionTypeRequest(
        aliases=[
            models.TransactionTypeAlias(
                type="StartingBalance",
                description="StartingBalance",
                transaction_class="StartingBalance",
                transaction_roles='None',
            )
        ],
        movements=movements2,
    )
)
prettyprint.print_transaction_type(response)

## 4) Create Cut Labels for Region Closes

In [ ]:
responses = global_fund_tools.create_cut_labels(
    api_factory=api_factory,
    exchange_names=["LSE", "NYSE"], 
    cut_label_type="market_close")

for response in responses:
    prettyprint.cut_label_response(response)

## 5) Load Close of Day Prices

In [ ]:
marketdata = pd.read_csv("data/global-fund-marketdata.csv")

marketdata['time'] = marketdata['time'].apply(lambda x: "LSE_market_close" if x=="LSEClose" else "NYSE_market_close")
marketdata['date_cutlabel'] = marketdata['date'] + 'N' + marketdata['time']

marketdata.head(n=30)

In [ ]:
marketdata_scopes = {
    'bloomberg': 'MarketData_Bloomberg_123',
    'reuters': 'MarketData_Reuters_123'
}

instrument_identifier_mapping = {
    'identifier_mapping': {
        "Figi": "figi",
        "Isin": "isin",
        "CurrencyPair": "currency"
    }
}

instrument_identifier_heirarchy = ["Figi", "Isin", "CurrencyPair"]

quotes_mapping_required = {
    "quote_type": "type",
    "effective_at": "date_cutlabel",
    "currency": "currency",
    "value": "price"
}
    
response_1 = global_fund_tools.upsert_quotes(
    api_factory=api_factory,
    scope=marketdata_scopes['bloomberg'],
    data_frame=marketdata.loc[marketdata['source'] == 'bloomberg'],
    instrument_identifier_mapping=instrument_identifier_mapping, 
    instrument_identifier_heirarchy=instrument_identifier_heirarchy, 
    required_mapping=quotes_mapping_required)


response_2 = global_fund_tools.upsert_quotes(
    api_factory=api_factory,
    scope=marketdata_scopes['reuters'],
    data_frame=marketdata.loc[marketdata['source'] == 'reuters'],
    instrument_identifier_mapping=instrument_identifier_mapping, 
    instrument_identifier_heirarchy=instrument_identifier_heirarchy, 
    required_mapping=quotes_mapping_required)


pd.concat([response_1, response_2], ignore_index=True)

## 6) Group Portfolios from the US & UK

In [ ]:
portfolio_group_scope = "Worldwide"

response = global_fund_tools.create_portfolio_group(
    api_factory=api_factory, 
    scope=portfolio_group_scope, 
    code="GlobalCreditFund", 
    portfolios=[
        models.ResourceId(
            scope=scopes["US"],
            code="GlobalCreditFund"),
        models.ResourceId(
            scope=scopes["UK"],
            code="GlobalCreditFund")
    ])

prettyprint.portfolio_group_response(response, "created")

## 7) Value the GlobalCreditFund for each Region Close

### LSE Close 11th September - Bloomberg Market Data

In [ ]:
global_fund_tools.valuation(
    api_factory=api_factory, 
    marketdata_scope=marketdata_scopes['bloomberg'], 
    portfolio_group=models.ResourceId(
        scope=portfolio_group_scope,
        code="GlobalCreditFund"),
    time="2019-09-11NLSE_market_close")

### LSE Close 11th September - Reuters Market Data

In [ ]:
global_fund_tools.valuation(
    api_factory=api_factory, 
    marketdata_scope=marketdata_scopes['reuters'], 
    portfolio_group=models.ResourceId(
        scope=portfolio_group_scope,
        code="GlobalCreditFund"),
    time="2019-09-11NLSE_market_close")

### NYSE Close 11th September - Reuters Market Data

In [ ]:
global_fund_tools.valuation(
    api_factory=api_factory, 
    marketdata_scope=marketdata_scopes['reuters'], 
    portfolio_group=models.ResourceId(
        scope=portfolio_group_scope,
        code="GlobalCreditFund"),
    time="2019-09-11NNYSE_market_close")

### NYSE Close 11th September - Bloomberg Market Data

In [ ]:
global_fund_tools.valuation(
    api_factory=api_factory, 
    marketdata_scope=marketdata_scopes['bloomberg'], 
    portfolio_group=models.ResourceId(
        scope=portfolio_group_scope,
        code="GlobalCreditFund"),
    time="2019-09-11NNYSE_market_close")